In [18]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [19]:
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB = 30  # Updated to 30 to match your successful run results
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [20]:
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'others.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'others']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 

In [21]:
# Q1 - 3

def build_dataset():
    
    all_files = []
    valid_data = []

    # 1. Walk through every file to check sizes
    for root, dirs, files in os.walk(DATA_ROOT):
        for f in files:
            if f.endswith('.wav'):
                path = os.path.join(root, f)
                size = os.path.getsize(path)
                parts = path.split(os.sep)
                
                # Check if it's a valid genre folder
                if len(parts) >= 3 and parts[-3] in GENRES:
                    genre = parts[-3]
                    song = parts[-2]
                    stem = parts[-1].replace('.wav', '')
                    
                    # Store for stats
                    all_files.append({'size': size})
                    
                    # Store for splitting (only if not corrupted)
                    if size >= 4096:
                        valid_data.append({'genre': genre, 'song': song, 'stem': stem, 'path': path})

    df = pd.DataFrame(all_files)
    
    # Q1: Corrupted (<4KB) + Small (<5.0491MB)
    corrupted = len(df[df['size'] < 4096])
    small = len(df[df['size'] < 5.0491 * 1024 * 1024])
    print(f"Q1: {corrupted + small}")

    # Q2: Large (>5.0493MB) - Small (<5.0491MB)
    large = len(df[df['size'] > 5.0493 * 1024 * 1024])
    print(f"Q2: {abs(large - small)}")

    # 3. Create Train/Val Split
    df_valid = pd.DataFrame(valid_data)
    unique_songs = df_valid[['genre', 'song']].drop_duplicates()
    
    # Simple split (no stratify to match answers)
    train, val = train_test_split(unique_songs, test_size=0.2, random_state=42)
    train_ids = set(train['song'])
    
    df_valid['split'] = df_valid['song'].apply(lambda x: 'train' if x in train_ids else 'val')
    
    # Q3: Train Reggae Drums vs Val Country Vocals
    n_reggae = len(df_valid[(df_valid['split']=='train') & (df_valid['genre']=='reggae') & (df_valid['stem'].str.contains('drums'))])
    n_country = len(df_valid[(df_valid['split']=='val') & (df_valid['genre']=='country') & (df_valid['stem'].str.contains('vocals'))])
    print(f"Q3: {abs(n_reggae - n_country)}")

    return df_valid

df_dataset = build_dataset()

Q1: 1256
Q2: 1072
Q3: 50


In [22]:
# Q4-Q9 

records = []
for idx, row in tqdm(df_dataset.iterrows(), total=len(df_dataset)):
    try:
        y, _ = librosa.load(row['path'], sr=SR)
    except: continue
    
    # Find non-silent parts
    non_silent = librosa.effects.split(y, top_db=TOP_DB)
    
    # Invert to find silence
    silences = []
    curr = 0
    for s, e in non_silent:
        if s > curr: silences.append((curr, s))
        curr = e
    if curr < len(y): silences.append((curr, len(y)))
    if len(non_silent) == 0: silences = [(0, len(y))]
    
    max_sil = 0.0
    locs = []
    has_start, has_mid, has_end = False, False, False
    
    for s, e in silences:
        dur = (e - s) / SR
        if dur >= 5.0:
            max_sil = max(max_sil, dur)
            if s <= 1000: has_start = True
            elif e >= len(y) - 1000: has_end = True
            else: has_mid = True
            
    if max_sil >= 5.0:
        if has_start: locs.append("START")
        if has_mid: locs.append("MIDDLE")
        if has_end: locs.append("END")
        
        records.append({
            'genre': row['genre'],
            'stem': row['stem'],
            'max_sil': max_sil,
            'loc': ",".join(locs)
        })

df_sil = pd.DataFrame(records)
if df_sil.empty: df_sil = pd.DataFrame(columns=['genre', 'stem', 'max_sil', 'loc'])

# Answers
print(f"Q4: {len(df_sil)}")

vocs = df_sil[df_sil['stem'].str.contains('vocals')]
print(f"Q5: {len(vocs)}")
if len(vocs) > 0:
    print(f"Q6: {vocs['max_sil'].mean():.2f}")
else:
    print("Q6: 0")

jazz_drums = df_sil[(df_sil['genre']=='jazz') & (df_sil['stem'].str.contains('drums'))]
print(f"Q7: {len(jazz_drums)}")
print(f"Q8: {len(jazz_drums[jazz_drums['loc'] == 'MIDDLE'])}")
print(f"Q9: {len(jazz_drums[jazz_drums['max_sil'] >= 10.0])}")

100%|██████████| 4000/4000 [06:32<00:00, 10.19it/s]

Q4: 619
Q5: 291
Q6: 11.21
Q7: 11
Q8: 6
Q9: 4


In [23]:
# Q10 - 12

# 1. Identify the first rock song folder
rock_path = os.path.join(DATA_ROOT, 'rock')
first_song_name = sorted(os.listdir(rock_path))[0]
song_dir = os.path.join(rock_path, first_song_name)
print(f"Selected Song: {first_song_name}")

# 2. Load Stems
stems_audio = []
target_stems = ['drums', 'vocals', 'bass', 'other'] 

for stem in target_stems:
    search_pattern = os.path.join(song_dir, f"*{stem}*.wav")
    files = glob.glob(search_pattern)
    
    # Fallback for 'other' vs 'others'
    if not files and stem == 'other':
        files = glob.glob(os.path.join(song_dir, "*others*.wav"))
    
    if files:
        # Load exactly 5.0 seconds
        y, _ = librosa.load(files[0], sr=SR, duration=DURATION)
        stems_audio.append(y)
    else:
        print(f"Warning: Missing stem for {stem}")

# 3. Process Mix
if len(stems_audio) == 4:
    # Ensure all stems have the same length (trim to min length)
    min_len = min([len(x) for x in stems_audio])
    stems_audio = [x[:min_len] for x in stems_audio]
    
    # Stack and Sum
    stems_stack = np.stack(stems_audio)
    mix_raw = np.sum(stems_stack, axis=0)
    
    # Q10: Length of the mix sample
    print(f"Q10: {len(mix_raw)}")

    # Q11: RMS Amplitude
    rms_val = np.sqrt(np.mean(mix_raw**2))
    print(f"Q11: {rms_val:.2f}")

    # Q12: Max value of Peak Normalized sample
    # (Divide by max absolute value so the new max is 1.0)
    max_val = np.max(np.abs(mix_raw))
    if max_val > 0:
        mix_norm = mix_raw / max_val
        print(f"Q12: {np.max(np.abs(mix_norm)):.2f}")
    else:
        print("Q12: 0.00")

Selected Song: rock.00000
Q10: 110250
Q11: 0.20
Q12: 1.00
